# LUNA demo — classifier-agnostic novelty detection

A toy example: a 3-class Gaussian problem with a distinct out-of-distribution
cluster. Any real pipeline works the same way — just feed LUNA your model's
`embeddings` (+ optional `logits`) as plain numpy arrays.

In [ ]:
import numpy as np
from luna import LUNAPipeline

rng = np.random.default_rng(0)
D = 32                                   # embedding dimension

def blob(center, n):
    return rng.normal(center, 1.0, size=(n, D)).astype(np.float32)

# --- in-distribution: 3 classes ---
centers = [rng.normal(0, 4, D) for _ in range(3)]
Ztr = np.vstack([blob(c, 400) for c in centers]);  ytr = np.repeat([0,1,2], 400)
Zva = np.vstack([blob(c, 150) for c in centers]);  yva = np.repeat([0,1,2], 150)

# --- out-of-distribution: a cluster far from every class ---
Zood = blob(rng.normal(9, 1, D), 120)

# fake logits: negative distance to each class center (what a classifier head would give)
def logits(Z):
    return -np.stack([np.linalg.norm(Z - c, axis=1) for c in centers], 1).astype(np.float32)

train = {"embeddings": Ztr, "logits": logits(Ztr), "labels": ytr}
val   = {"embeddings": Zva, "logits": logits(Zva), "labels": yva}
ood   = {"embeddings": Zood, "logits": logits(Zood)}

In [ ]:
# Fit the 16 scorers on in-distribution statistics and score anything
pipe = LUNAPipeline(combiner="lightgbm", target_far=0.01)
pipe.fit(train)

s_val = pipe.score(val)
s_ood = pipe.score(ood)
print("active scores:", pipe.active_methods)

In [ ]:
# Per-family AUROC + detections at 1% FAR (no outlier labels used anywhere)
fam = pipe.family_analysis(val, ood)
for name, r in fam.items():
    print(f"{name:12s} AUROC={r['auroc']:.3f}  detected {r['detected']}/{r['total']}")

In [ ]:
# Single-score sanity plot: within-class Mahalanobis separates ID from OOD
import matplotlib.pyplot as plt
plt.hist(s_val["mahal_within"], bins=40, alpha=.6, label="in-distribution (val)")
plt.hist(s_ood["mahal_within"], bins=40, alpha=.6, label="OOD")
plt.xlabel("within-class Mahalanobis"); plt.ylabel("count"); plt.legend(); plt.show()

### Supervised combiner (optional)

If you have a *small* set of known outliers, `fit_combiner` trains the LightGBM
combination of all 16 scores and `predict` returns combined scores + flags at the
target false-alarm rate. When *evaluating* detection on known outliers, use a
k-fold protocol so no outlier is scored by a combiner that trained on it.

In [ ]:
expo, test = ood["embeddings"][:60], ood["embeddings"][60:]
pipe.fit_combiner(val, {"embeddings": expo, "logits": logits(expo)})
out = pipe.predict({"embeddings": test, "logits": logits(test)})
print(f"flagged {out['flags'].sum()}/{len(test)} held-out OOD at {pipe.target_far:.0%} FAR")